# uniCOIL (2021)
---
[[paper]](https://arxiv.org/abs/2106.14807)<br>
uniCOIL = Universal Contextualized Inverted List

__uniCOIL__ — это метод <u>обучаемого</u> разреженного поиска (Learned Sparse Retrieval), который оптимизирует веса терминов в инвертированном индексе с помощью нейросетевых моделей. Он позволяет эффективно решать проблему семантического разрыва (vocabulary mismatch) и учитывать контекстуальную важность слов, сохраняя при этом возможность использования стандартных поисковых движков вроде Lucene.

__Завдача__<br>
Поиск наиболее релевантных документов в больших коллекциях. На входе имеем текстовый запрос $q$, на выходе — ранжированный список документов $d$, где оценка релевантности вычисляется как скалярное произведение весов общих терминов (Exact Match).

__Мотивация__<br>
Существовал значительный разрыв между эффективностью и качеством. Dense Retrieval (например, DPR, 2020) отлично справляется с семантикой, но требует больших вычислительных ресурсов для хранения векторного индекса и поиска по нему (HNSW/FAISS). Sparse Retrieval (BM25) очень быстр и масштабируем, но работает только с точными совпадениями слов и использует примитивные статистики (TF-IDF). Предшественник метода — модель COIL (2021) — пыталась совместить подходы, сохраняя в инвертированном индексе небольшие векторы для каждого слова, что усложняло архитектуру поискового движка. Авторы uniCOIL стремились упростить эту схему до классического инвертированного индекса, где для каждого слова хранится только одно число (скаляр), не теряя в качестве.

__Альтернативы__<br>
На момент появления uniCOIL основными конкурентами были:
- BM25: использует статические формулы частотности, не учитывает контекст.
- DeepCT (2019): предсказывает важность слова в контексте документа, но плохо работает с запросами и не решает проблему отсутствующих слов (expansion).
- Doc2query-T5 (2019): генерирует новые слова для документа, расширяя его семантику, но использует стандартные веса BM25.
- SPLADE (2021): мощный метод LSR, использующий Sparse Regularization (L1), но обладающий более сложным процессом обучения и более "тяжелыми" (менее разреженными) представлениями.

__Идея__<br>
Идея заключается в переходе от векторов (как в COIL) к скалярным весам, которые вычисляются контекстуально. uniCOIL объединяет два процесса: Term Weighting (определение важности слова в контексте) и Document Expansion (добавление новых релевантных слов). В отличие от классических методов, здесь веса терминов — это не частота появления, а выученная моделью оценка вклада этого слова в релевантность.

__Архитектура__<br>
Модель строится на базе BERT (обычно `distilbert-base-uncased` для скорости).
1.  Encoder: BERT обрабатывает текст (запрос или документ) и выдает последовательность контекстуализированных векторов $h_i$ для каждого токена.
2.  Weighting Layer: Поверх каждого вектора $h_i$ применяется полносвязный слой с активацией ReLU, который проецирует вектор в одно скалярное значение $w_i = \text{ReLU}(w^T h_i + b)$.
3.  Representation: Представление документа/запроса — это вектор в пространстве словаря (Vocabulary-sized sparse vector), где ненулевые значения стоят только в позициях токенов, присутствующих в тексте.
4.  Expansion: Для документов используется предварительный этап генерации дополнительных токенов (через doc2query-T5), после чего uniCOIL назначает им веса.

__Обучение__<br>
Обучение проходит по принципу Siamese Network (но с разными весами или стратегиями для запроса и документа):
1.  Loss function: Используется Contrastive Loss (обычно Cross-Entropy над оценками релевантности). Модель учится максимизировать скалярное произведение $\sum w_q \cdot w_d$ для релевантных пар и минимизировать для нерелевантных.
2.  Data: В качестве обучающей выборки используется MS MARCO. Важным элементом является использование Hard Negatives (документов, которые BM25 считает релевантными, но которые на самом деле ими не являются).
3.  Distillation: Часто применяется Knowledge Distillation из более мощного Cross-Encoder (например, на базе ELECTRA), что дает прирост в несколько пунктов метрики nDCG.

__Инференс__<br>
1.  Offline (Индексация): Документы расширяются через doc2query, прогоняются через uniCOIL, полученные веса $w_i$ квантуются в целые числа и сохраняются в стандартный инвертированный индекс (например, в поле частоты слова в Lucene).
2.  Online (Поиск): Запрос $q$ токенизируется, проходит через BERT-энкодер uniCOIL для получения весов терминов запроса.
3.  Retrieval: Выполняется обычный Weighted Boolean Query в поисковом движке. Финальный скор — это просто сумма весов совпавших токенов.

__Результаты__<br>
Сравнение проводилось на MS MARCO Passage Ranking и наборе тестов BEIR:
- uniCOIL показал MRR@10 равный 0.351 на MS MARCO, что на 16пп выше, чем у классического BM25 (0.184) и сопоставимо с более сложными Dense моделями того времени.
- По сравнению с оригинальным COIL, uniCOIL уменьшил размер индекса в 5-10 раз (за счет хранения одного скаляра вместо вектора из 32 компонентов), при этом сохранив аналогичную точность.
- В отличие от Dense Retrieval, uniCOIL не требует GPU на этапе поиска и работает на стандартной CPU-инфраструктуре со скоростью, близкой к классическому полнотекстовому поиску.

## 📝 Критический анализ

```markdown
# uniCOIL (2021)
---
[[paper]](https://arxiv.org/abs/2106.14807)<br>
uniCOIL = Universal Contextualized Inverted List

__uniCOIL__ — метод **Learned Sparse Retrieval**, оптимизирующий веса терминов в инвертированном индексе с помощью нейросетей. Он решает проблему семантического разрыва и учитывает контекстуальную важность слов, сохраняя совместимость с поисковыми движками, такими как Lucene.

__Задача__<br>
Поиск релевантных документов в больших коллекциях. На входе текстовый запрос $q$, на выходе — ранжированный список документов $d$, где релевантность оценивается через скалярное произведение весов терминов.

__Мотивация__<br>
Dense Retrieval, как DPR (2020), требует значительных ресурсов, а Sparse Retrieval, как BM25, ограничен точными совпадениями. Предшественник COIL (2021) усложнял архитектуру, храня векторы для каждого слова. uniCOIL упрощает схему, используя скалярные веса.

__Альтернативы__<br>
- BM25: статические частотные формулы.
- DeepCT (2019): учитывает контекст, но не решает проблему отсутствующих слов.
- Doc2query-T5 (2019): расширяет семантику, но использует стандартные веса.
- SPLADE (2021): сложный процесс обучения и менее разреженные представления.

__Идея__<br>
Переход от векторов к скалярным весам, вычисляемым контекстуально. uniCOIL объединяет Term Weighting и Document Expansion, где веса терминов — это оценка вклада слова в релевантность.

__Архитектура__<br>
<img src="img/img.png" width=500>
1. **Encoder**: BERT обрабатывает текст и выдает векторы $h_i$ для токенов.
2. **Weighting Layer**: Применяется полносвязный слой с ReLU, проецирующий вектор в скаляр $w_i$.
3. **Representation**: Документ/запрос представлен вектором, где ненулевые значения только для присутствующих токенов.
4. **Expansion**: Генерация дополнительных токенов через doc2query-T5 с назначением весов.

__Обучение__<br>
Используется Contrastive Loss для максимизации скалярного произведения релевантных пар. Обучение на MS MARCO с Hard Negatives и Knowledge Distillation из Cross-Encoder.

__Инференс__<br>
1. **Offline**: Документы расширяются, прогоняются через uniCOIL, веса квантуются и сохраняются в инвертированный индекс.
2. **Online**: Запрос токенизируется, проходит через BERT-энкодер для получения весов.
3. **Retrieval**: Выполняется Weighted Boolean Query, финальный скор — сумма весов совпавших токенов.

__Результаты__<br>
- MRR@10 на MS MARCO: 0.351, на 16пп выше BM25 (0.184), сопоставимо с Dense моделями.
- uniCOIL уменьшил размер индекса в 5-10 раз по сравнению с COIL, сохранив точность.
- Работает на CPU-инфраструктуре со скоростью, близкой к классическому полнотекстовому поиску.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования uniCOIL для обработки текстового запроса и документа.
# Мы будем использовать библиотеку transformers для работы с BERT.

from transformers import DistilBertTokenizer, DistilBertModel
import torch
import torch.nn as nn
import torch.nn.functional as F

# Инициализация токенизатора и модели BERT
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Пример текста запроса и документа
query = "What is the capital of France?"
document = "Paris is the capital city of France."

# Токенизация текста
query_tokens = tokenizer(query, return_tensors='pt')
document_tokens = tokenizer(document, return_tensors='pt')

# Получение контекстуализированных векторов BERT для запроса и документа
query_outputs = model(**query_tokens)
document_outputs = model(**document_tokens)

# Векторы скрытых состояний для каждого токена
query_hidden_states = query_outputs.last_hidden_state
document_hidden_states = document_outputs.last_hidden_state

# Полносвязный слой для вычисления весов терминов
class TermWeightingLayer(nn.Module):
    def __init__(self, hidden_size):
        super(TermWeightingLayer, self).__init__()
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Применение полносвязного слоя и ReLU
        return F.relu(self.fc(x))

# Инициализация слоя для вычисления весов
term_weighting_layer = TermWeightingLayer(hidden_size=query_hidden_states.size(-1))

# Вычисление весов терминов для запроса и документа
query_weights = term_weighting_layer(query_hidden_states).squeeze(-1)
document_weights = term_weighting_layer(document_hidden_states).squeeze(-1)

# Пример вычисления скалярного произведения весов для оценки релевантности
# Мы используем только те токены, которые присутствуют в обоих текстах
common_tokens = set(tokenizer.convert_ids_to_tokens(query_tokens['input_ids'][0])) & \
                set(tokenizer.convert_ids_to_tokens(document_tokens['input_ids'][0]))

# Суммируем веса только для общих токенов
relevance_score = sum(query_weights[i] * document_weights[j]
                      for i, token_q in enumerate(tokenizer.convert_ids_to_tokens(query_tokens['input_ids'][0]))
                      for j, token_d in enumerate(tokenizer.convert_ids_to_tokens(document_tokens['input_ids'][0]))
                      if token_q == token_d and token_q in common_tokens)

print(f"Relevance score: {relevance_score.item()}")

# Этот пример иллюстрирует, как uniCOIL использует контекстуализированные векторы BERT
# для вычисления весов терминов и оценки релевантности через скалярное произведение.
# В отличие от традиционных методов, веса терминов здесь обучаются, что позволяет
# учитывать контекст и семантику, сохраняя при этом эффективность разреженного поиска.